# Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from nedis.parallelization import set_threads_for_external_libraries
set_threads_for_external_libraries(16)

In [ ]:
import sys
import logging
import re

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import seaborn as sns

import scipy
import sklearn.cluster
import pathlib


In [ ]:
import nedis.data.synthetic
import nedis.data.tasks

from nedis.experiments import load_component, log_cluster_experiment
from nedis.utils import slugify

# Setup

In [ ]:
%run -m rpy2.situation

In [ ]:
logging.basicConfig(stream=sys.stdout)
logging.getLogger("nedis").setLevel(level=logging.DEBUG)

# Load Data

In [ ]:
%%bash
# Downloading the preeclampsia dataset from Zenodo
DATA_DIR=../data/raw/preeclampsia
mkdir -p $DATA_DIR
wget https://zenodo.org/records/10545240/files/preeclampsia.rda -O ${DATA_DIR}/preeclampsia.rda

In [ ]:
data_path = "../data"
data_path

In [ ]:
task_list_complete = nedis.data.tasks.load_task_list(
    data_path=data_path,
    include=["pree"])

In [ ]:
print("Available tasks")
for i, t in enumerate(task_list_complete):
    print(f"{i:2}: {t['name']:60} [{t['id']:60}]: {t['features'].shape[1]}")

# Parameters

In [ ]:
# parameters (papermill)
task_id = "preeclampsia-ref-all-all"
transformer = "default"
handle_heteroscedasticity = "robust"
normalize_timepoints = "robust"
multi_sample=True
random_state = 120
# random_state = 122

In [ ]:
# set task
if isinstance(task_id, int):
    task = task_list_complete[task_id]
else:
    task = [t for t in task_list_complete if task_id == t['id']]
    assert len(task) == 1, f"{task_id}, {[t['name'] for t in task]}"
    task = task[0]
print(task["name"])

In [ ]:
# set experiment name
task_name = slugify(task['name'])
exp_name = f"classification___" \
    + f"___params" \
    + f"___transformer_{transformer}" \
    + f"___task_{task_name}" \
    + f"___handle-heteroscedasticity_{handle_heteroscedasticity}"

if normalize_timepoints is not None:
    exp_name += f"___normalize_timepoints_{normalize_timepoints}"
    
if not multi_sample:
    exp_name += f"___multi-sample_{multi_sample}"

exp_name += f"___random-state_{random_state}"
    

print("Experiment:")
print(exp_name)

# Prepare task

In [ ]:
# visualize gestational age distribution
fig, ax = plt.subplots(figsize=(8,4))
sns.histplot(x=task["timepoints_continuous"], hue=task["timepoints"], bins=np.arange(7, 29, 1), element="step", palette="Set2", ax=ax)
for tp in np.unique(task["timepoints"]):
    plt.axvline(np.median(task["timepoints_continuous"][task["timepoints"] == tp]), linestyle="--", color="gray")
    mean = np.mean(task["timepoints_continuous"][task["timepoints"] == tp])
    std = np.std(task["timepoints_continuous"][task["timepoints"] == tp])
    ax.annotate(
        f"{mean:.2f}±{std:.2f}", 
        xy=(mean, 0), 
        xytext=(0,10), 
        textcoords="offset points", 
        ha="left")

ax.set_xlabel("Gestational age (weeks)")
ax.set_ylabel("Number of samples")

output_dir = pathlib.Path("../_out/paper")
output_dir.mkdir(exist_ok=True, parents=True)
fig.savefig(output_dir / f"task_{task_name}___gestational_age_distribution.pdf", dpi=300, bbox_inches="tight")

In [ ]:
X = task["features"]
y = task["groups"]
entities = task["entities"]
timepoints = task["timepoints"]
feature_names = task["feature_names"]

y_unique = np.unique(y)

y_map = {yy:i for i,yy in enumerate(y_unique)}
y = np.array([y_map[yy] for yy in y])
y_unique = np.unique(y)

print(task["name"])
print(X.shape)
print(y_unique)

In [ ]:
# normalize each condition
if handle_heteroscedasticity is None:
    pass

elif isinstance(handle_heteroscedasticity, str):
    
    if handle_heteroscedasticity == "normalize":
        scaler = sklearn.preprocessing.StandardScaler()
    elif handle_heteroscedasticity == "robust":
        scaler = sklearn.preprocessing.RobustScaler()
    else:
        raise ValueError(f"Unknown timepoint normalization: {normalize_timepoints}")
    
    y_unique = np.unique(y)
    X = X.copy()
    for yy in y_unique:
        X[y == yy] = scaler.fit_transform(X[y == yy])

elif isinstance(handle_heteroscedasticity, float):
    
    import nedis.cordis.filtering
    
    heteroscedasticity_filter = nedis.cordis.filtering.HeteroscedacticityFilter(
        test_mean='kruskal',
        test_var='levene',
        p_threshold=handle_heteroscedasticity)
    
    msk = heteroscedasticity_filter.get_feature_mask(
        X, 
        subset_masks=np.array([(yy == y) for yy in y_unique]).transpose()
    )
    X = X[:, msk].copy()
    feature_names = feature_names[msk]

In [ ]:
if normalize_timepoints is not None:
    if normalize_timepoints == "normalize":
        scaler = sklearn.preprocessing.StandardScaler()
    elif normalize_timepoints == "robust":
        scaler = sklearn.preprocessing.RobustScaler()
    else:
        raise ValueError(f"Unknown timepoint normalization: {normalize_timepoints}")
    t_unique = np.unique(timepoints)
    X = X.copy()
    for t in t_unique:
        X[timepoints == t] = scaler.fit_transform(X[timepoints == t])

# Run correlation disruption discovery

In [ ]:
fit_params = dict(
    X=X,
    y=y,
    subset_masks="y",
    samples=entities if multi_sample else None
)

In [ ]:
X[timepoints == 3].shape

In [ ]:
def transformer_default():
    
    from nedis.cordis.disruption import CorrelationDisruption
    nedis
    from nedis.cluster.leidenalg import WeightedLeidenClustering
    from nedis.cordis.clustering import ReferenceCorrelationMatrixClusteringStep
    nedis
    from nedis.cordis.optimization import GreedyRefinementClusterOptimization, SequentialOptimizationStep

    clustering_algorithm = WeightedLeidenClustering(random_state=random_state)

    clustering_step = ReferenceCorrelationMatrixClusteringStep(
        clustering_algorithm=clustering_algorithm,
    #     clustering_absolute_correlation=True,
    #     correlation_function="spearman",
    #     feature_filters=None
    )

    rowcol_optimization = GreedyRefinementClusterOptimization(
        separation_score="auc",
    #     separation_score_comparison='all',
        refinement_mode="features",
    #     correlation_function="spearman",
    #     disruption_metric="direction",
    #     disruption_robustness='loo',
    #     disruption_aggregation='mean',
    #     max_runs=-1
    )

    optimization_step = SequentialOptimizationStep([
            rowcol_optimization
        ],
        max_runs=1)

    t = CorrelationDisruption(
        clustering_step=clustering_step,
        cluster_optimization_step=optimization_step,
        filter_coverage_threshold=0.5,
#         separation_score_threshold=("auto", 1)
    )
    t.fit(**fit_params)
    
    return t

In [ ]:
def transformer_edges():
    
    import sklearn.cluster
    from nedis.cordis.disruption import CorrelationDisruption
    
    from nedis.cordis.clustering import CorrelationProfileClusteringStep
    from nedis.cordis.optimization import GreedyRefinementClusterOptimization, SequentialOptimizationStep

    clustering_algorithm = sklearn.cluster.KMeans(n_clusters=256, random_state=random_state)

    clustering_step = CorrelationProfileClusteringStep(
        clustering_algorithm=clustering_algorithm,
    #     clustering_absolute_correlation=True,
    #     correlation_function="spearman",
    #     feature_filters=None
    )

    rowcol_optimization = GreedyRefinementClusterOptimization(
        separation_score="spearman",
    #     separation_score_comparison='all',
        refinement_mode="features",
    #     correlation_function="spearman",
    #     disruption_metric="direction",
    #     disruption_robustness='loo',
    #     disruption_aggregation='mean',
    #     max_runs=-1
    )

    optimization_step = SequentialOptimizationStep([
            rowcol_optimization, 
        ],
        max_runs=1)

    t = CorrelationDisruption(
        clustering_step=clustering_step,
        cluster_optimization_step=optimization_step,
        filter_coverage_threshold=0.5, 
#         separation_score_threshold=("auto", 1)
    )
    t.fit(**fit_params)
    
    return t

In [ ]:
t = load_component(
    "transformer",
    func=globals()["transformer_" + transformer],
    output_dir="../_out/realworld/" + exp_name,
    overwrite=False
)

# Log experiments

In [ ]:
log_cluster_experiment(
    exp_name,
    X, y, entities, feature_names, transformer=t,
    multi_sample=multi_sample,
    y_map={v:k for k,v in y_map.items()},
    include_stats=["correlation-disruption"],
    exclude_stats=["overall"],
    cluster_summary_visualizations=["kde", "box"],
    output_dir="../_out/realworld",
    random_state=random_state,
    show_plots=False)

# Cluster visualization

In [ ]:
clusters = sorted(t.clusters_, key=lambda c: -c["reference_score"])[:6]

In [ ]:
exp_dir = pathlib.Path("../_out/realworld") / exp_name

In [ ]:
coordinates_dict = load_component("coordinates_dict", output_dir=exp_dir)

In [ ]:
correlation_matrices_dict = load_component("correlation_matrices_dict", output_dir=exp_dir)

In [ ]:
disruption_matrices_dict = load_component("disruption_matrices_dict", output_dir=exp_dir)

In [ ]:
timepoint_map_pp = {
    0: "Healthy", 1: "Preeclamptic"
}

In [ ]:
def format_ax(ax):
    # https://stackoverflow.com/a/27361819/991496

    # Hide the right and top spines
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)

    # Only show ticks on the left and bottom spines
    ax.yaxis.set_ticks_position('left')
    ax.xaxis.set_ticks_position('bottom')

In [ ]:
output_dir = pathlib.Path("../_out/paper/clusters")
output_dir.mkdir(exist_ok=True, parents=True)

In [ ]:
from nedis.cordis.utils import prepare_y
yy = prepare_y(y, entities)

In [ ]:
from nedis.visualization import nx_plot
from matplotlib.colors import Normalize

cmap = sns.color_palette(palette="vlag", as_cmap=True)

def plot_results(i_cluster, cluster, show_plots=False):

    print(cluster["id"], cluster["reference_score"], len(cluster["rows"]))   

    correlation_threshold = -1
    
    correlation_matrix = correlation_matrices_dict[cluster["reference_label"]]
    # correlation_matrix = correlation_matrices_dict[0]
    coo = coordinates_dict[cluster["reference_label"]]
    coo = coordinates_dict[0]
    disruption_matrices = disruption_matrices_dict[cluster["reference_label"]]
    dis = np.max([np.max(np.abs(np.median(d, axis=0))) for d in disruption_matrices_dict.values()])
    norm = plt.Normalize(-dis, dis)

    nodes_pos = {
        i:p 
        for i, p in enumerate(coo)
        if i in cluster["rows"] or i in cluster["columns"]}
    edges = [
        (src, dst) for src, dst in zip(*cluster["edges"].nonzero())
        if src != dst and 
            abs(correlation_matrix[src, dst]) >= correlation_threshold]
    rows = list({src for src, _ in edges})
    columns = list({dst for _, dst in edges})
    
    g = sns.clustermap(correlation_matrix[cluster["rows"],:][:,cluster["rows"]])
    idx_rows, idx_cols = g.dendrogram_row.reordered_ind, g.dendrogram_col.reordered_ind
    plt.close()
    
    n_rows = 2
    scale = 1
    # fig, axes = plt.subplots(
    #     n_rows, 4,
    #     figsize=(4 * 4 * scale, n_rows * 4 * scale), 
    #     dpi=300,
    #     squeeze=False)
    # fig.tight_layout(h_pad=5, w_pad=3)

    fig = plt.figure(
        figsize=(4 * 4 * scale, n_rows * 4 * scale), 
        dpi=200,
        constrained_layout=False
        )
    gs1 = fig.add_gridspec(
        nrows=1, ncols=13, 
        left=0.00, right=1.00,
        top=1, bottom=0.57,
        wspace=0.8)
    ax_r1_cb = fig.add_subplot(gs1[0, 0])
    ax1 = fig.add_subplot(gs1[0, 1:4])
    ax2 = fig.add_subplot(gs1[0, 6:9])
    ax3 = fig.add_subplot(gs1[0, 10:13])

    gs2 = fig.add_gridspec(
        nrows=1, ncols=13, 
        left=0.00, right=1.00,
        top=0.43, bottom=0,
        wspace=0.8)
    ax_r2 = [
        fig.add_subplot(gs2[0, 1 + i*3: 1+i*3+3])
        for i in range(4)
    ]
    ax_r2_cb = fig.add_subplot(gs2[0, 0])
    
    ax = ax1

    def nodes_node_size(g, n, d):
        if n in rows and n in columns:
            return 100
        else:
            return 0.1

    def nodes_node_color(g, n, d):
        if n in rows and n in columns:
            return "black"
        else:
            return "grey"

    def edges_width(g, src, dst, d):
        abscor = abs(correlation_matrix[src, dst])
        if src == dst and not np.isclose(correlation_matrix[src,dst], 1):
            print(correlation_matrix[src,dst])
        t = 0.1
        if abscor < t:
            return 0
        else:
            return (abscor - t) / (1 - t) * 5

    def edges_edge_color(g, src, dst, d):
        cmap = sns.color_palette(palette="vlag", as_cmap=True)
        cor = correlation_matrix[src, dst]
        c = (cor + 1) / 2
        color = cmap(c)
        color = (*color[:-1], 0.7)
        return color
    
    ax.scatter(
        coo[:,0], coo[:,1], 
        color=[.5,.5,.5,.3], 
        linewidths=0, 
        edgecolors=None)
    
    nx_plot(
        # nodes=np.arange(coo.shape[0]),
        nodes_pos=nodes_pos,
        nodes_args=dict(
            node_size=nodes_node_size,
            node_color=nodes_node_color,
            edgecolors="white"
        ),
        edges=edges,
        edges_args=dict(
            width=edges_width,
            edge_color=edges_edge_color,
        ),
        nodes_labels=np.arange(len(feature_names)),
        nodes_labels_args={
            "font_size": 6,
            "textcoords": 'offset points',
            "xytext": (5,0),
            "color": "black"
        },
        ax=ax)
    ax.set_title("A. Correlation network: " + timepoint_map_pp[cluster["reference_label"]], loc="left")
    ax.axis("off")
    
    ax = ax3
    ax.set_title("C. Disruption profile", loc='left')

    dis_values = disruption_matrices_dict[cluster["reference_label"]]
    dis_values = np.mean(dis_values[:,cluster["edges"].toarray() == 1], axis=(1))

    sns.boxplot(
        x=yy, 
        y=dis_values, 
        # color=sns.color_palette()[0],
        # # groups=entities,
        # groups_kwargs=dict(color="grey", linewidth=1, alpha=0.2),
        ax=ax)

    format_ax(ax)
    ax.set(xlabel="timepoint", ylabel="disruption", xticklabels=["healthy", "preeclamptic"])
    
#     ax = axes[row, 2]; ax.axis("off")
#     ax.set_title("B. Module features", loc='left')
#     offset = 0
#     for i, t in np.array(list(enumerate(feature_names)))[cluster["rows"]][idx_rows]:
#             ax.annotate(f"{int(i):3d}. {re.sub('^[0-9]* ', '', t.replace('_', ' '))}", (0, - offset), fontsize=5)
#             offset += 1
#     ax.set_ylim([-offset, 0])
    
    ax = ax2;
    ax.set_title("B. Correlation heatmap: " + timepoint_map_pp[cluster["reference_label"]], loc='left')
    sns.heatmap(
        correlation_matrices_dict[cluster["reference_label"]][cluster["rows"],:][:,cluster["columns"]][idx_rows,:][:,idx_cols],
        center=0, vmin=-1, vmax=1, cmap=cmap,
        xticklabels=np.array(feature_names)[cluster["rows"]][idx_rows],
        yticklabels=np.array([f"{i}_{f}" for i,f in enumerate(feature_names)])[cluster["rows"]][idx_rows],
        cbar=False,
        ax=ax)
    
    ticklabels = []
    for l in ax.get_yticklabels():
        arr = l.get_text().split("_")
        ticklabels.append(f"{int(arr[0]):3d}. {' '.join(arr[1:]):>37s}")
    ax.set_yticklabels(ticklabels, fontsize = 5, family="monospace")
    ax.set_xticklabels(["" for _ in ax.get_xticklabels()], fontsize = 5)

    from matplotlib import cm
    ax = ax_r1_cb; ax.axis("off")
    fig.colorbar(cm.ScalarMappable(norm=Normalize(-1,1), cmap="vlag"), ax=ax, label="correlation", pad=0.3, location='left', fraction=1)
    ax = ax_r2_cb; ax.axis("off")
    fig.colorbar(cm.ScalarMappable(norm=norm, cmap="vlag"), ax=ax, label="disruption", pad=0.3, location='left', fraction=1)
        
    
    ax.annotate("D. Disruption networks", (0.1,0.50), xycoords="figure fraction", ha="left", fontsize=12)
    ax.annotate("E. Disruption heatmaps", (0.57,0.50), xycoords="figure fraction", ha="left", fontsize=12)
    
#     # sns.heatmap(
#     #     correlation_matrices_dict[cluster["reference_label"]][cluster["rows"],:][:,cluster["columns"]][idx_rows,:][:,idx_cols],
#     #     center=0, vmin=-1, vmax=1, cmap=cmap,
#     #     ax=ax)
    
    # for i, (key, _) in enumerate(coordinates_dict.items()):
    for i, key in enumerate([0, 1]):
        
        ax = ax_r2[i]
        ax.set_title(timepoint_map_pp[key])
        
        ax.axis("off")
        ax.scatter(coo[:,0], coo[:,1], color=[.5, .5, .5, .3])

        def edges_width(g, src, dst, d):
            dis = np.median(disruption_matrices[yy == key, src, dst])
            if True:\
                return np.abs((norm(dis) - 0.5)) * 2 * 10
            else:
                return .5

        def edges_edge_color(g, src, dst, d):
            dis = np.median(disruption_matrices[yy == key, src, dst])
            color = cmap(norm(dis))
            alpha = min(1, max(0, np.abs((norm(dis) - 0.5)) * 2))
            color = (*color[:-1], alpha)

            if True:
                return color
            else:
                return [0.7, 0.7, 0.7, 0.05]   

        nx_plot(
            # nodes=np.arange(coo.shape[0]),
            nodes_pos=nodes_pos,
            nodes_args=dict(
                node_size=nodes_node_size,
                node_color=nodes_node_color,
                edgecolors="white"
            ),
            edges=edges,
            edges_args=dict(
                width=edges_width,
                edge_color=edges_edge_color,
            ),
            # nodes_labels=labels,
            nodes_labels_args={
                "font_size": 2
            },
            ax=ax)
        
        
        # ax = axes[r, i]; r += 1
        # sns.heatmap(
        #     correlation_matrices_dict[key][cluster["rows"],:][:,cluster["columns"]][idx_rows,:][:,idx_cols],
        #     center=0, vmin=-1, vmax=1, cmap=cmap,
        #     xticklabels=cluster["rows"][idx_rows],
        #     yticklabels=cluster["rows"][idx_rows],
        #     ax=ax)
        
        ax = ax_r2[i + 2]
        ax.set_title(timepoint_map_pp[key])
        
        mean = np.mean(disruption_matrices[key == yy], axis=0)[cluster["rows"],:][:,cluster["columns"]]
        sns.heatmap(
            mean[idx_rows,:][:,idx_cols],
            center=0, vmin=-dis, vmax=dis, cmap=cmap,
            xticklabels=cluster["rows"][idx_rows],
            yticklabels=cluster["rows"][idx_rows],
            cbar=False,
            # cbar=(i == 3),
            ax=ax)
        
        # ax.set(xticklabels=np.arange(len(feature_names))[cluster["rows"]])
        ax.set_xticklabels(ax.get_xticklabels(), fontsize = 7)
        for tick in ax.xaxis.get_major_ticks()[1::2]:
            tick.set_pad(18)
            
        if i == 0:
            ax.set_yticklabels(ax.get_yticklabels(), fontsize = 7)
        else:
            ax.set_yticklabels(["" for _ in ax.get_yticklabels()], fontsize = 7)
        for tick in ax.yaxis.get_major_ticks()[1::2]:
            tick.set_pad(18)
        ax.set_xticklabels(["" for _ in ax.get_xticklabels()], fontsize = 5)
    
    fig.savefig(output_dir / f"cluster_classification_{i_cluster}.png", bbox_inches="tight")
    fig.savefig(output_dir / f"cluster_classification_{i_cluster}.pdf", bbox_inches="tight")

    if show_plots:
        plt.show()
        plt.tight_layout()
    else:
        plt.close()

for i_cluster, cluster in enumerate(clusters[:]): 
    plot_results(i_cluster, cluster, show_plots=True)

In [ ]:
from nedis.visualization import nx_plot
from matplotlib.colors import Normalize

cmap = sns.color_palette(palette="vlag", as_cmap=True)

def plot_results(i_cluster, cluster, show_plots=False):

    print(cluster["id"], cluster["reference_score"], len(cluster["rows"]))   

    correlation_threshold = -1
    
    correlation_matrix = correlation_matrices_dict[cluster["reference_label"]]
    # correlation_matrix = correlation_matrices_dict[0]
    coo = coordinates_dict[cluster["reference_label"]]
    coo = coordinates_dict[0]
    disruption_matrices = disruption_matrices_dict[cluster["reference_label"]]
    dis = np.max([np.max(np.abs(np.median(d, axis=0))) for d in disruption_matrices_dict.values()])
    norm = plt.Normalize(-dis, dis)

    nodes_pos = {
        i:p 
        for i, p in enumerate(coo)
        if i in cluster["rows"] or i in cluster["columns"]}
    edges = [
        (src, dst) for src, dst in zip(*cluster["edges"].nonzero())
        if src != dst and 
            abs(correlation_matrix[src, dst]) >= correlation_threshold]
    rows = list({src for src, _ in edges})
    columns = list({dst for _, dst in edges})
    
    g = sns.clustermap(correlation_matrix[cluster["rows"],:][:,cluster["rows"]])
    idx_rows, idx_cols = g.dendrogram_row.reordered_ind, g.dendrogram_col.reordered_ind
    plt.close()
    
    n_rows = 2
    scale = 1
    # fig, axes = plt.subplots(
    #     n_rows, 4,
    #     figsize=(4 * 4 * scale, n_rows * 4 * scale), 
    #     dpi=300,
    #     squeeze=False)
    # fig.tight_layout(h_pad=5, w_pad=3)

    fig = plt.figure(
        figsize=(4 * 4 * scale, n_rows * 4 * scale), 
        dpi=200,
        constrained_layout=False
        )
    gs1 = fig.add_gridspec(
        nrows=1, ncols=13, 
        left=0.00, right=1.00,
        top=1, bottom=0.57,
        wspace=0.8)
    ax_r1_cb = fig.add_subplot(gs1[0, 0])
    ax1 = fig.add_subplot(gs1[0, 1:4])
    ax2 = fig.add_subplot(gs1[0, 6:9])
    ax3 = fig.add_subplot(gs1[0, 10:13])

    gs2 = fig.add_gridspec(
        nrows=1, ncols=13, 
        left=0.00, right=1.00,
        top=0.43, bottom=0,
        wspace=0.8)
    ax_r2 = [
        fig.add_subplot(gs2[0, 1 + i*3: 1+i*3+3])
        for i in range(4)
    ]
    ax_r2_cb = fig.add_subplot(gs2[0, 0])
    
    ax = ax1

    def nodes_node_size(g, n, d):
        if n in rows and n in columns:
            return 100
        else:
            return 0.1

    palette = [
        "#4E79A7",  # Blue
        "#F28E2B",  # Orange
        # "#E15759",  # Red
        # "#76B7B2",  # Turquoise
        "#EDC948",  # Gold
        "#B07AA1",  # Purple
        "#59A14F",  # Green
        # "#FF9DA7",  # Pink
    ] 

    palette = [
        "#7fc97f",
        "#beaed4",
        "#fdc086",
        "#ffff99",
        "#386cb0"
    ]

    # palette = [ # waaaaaaay to bright
    #     "#b3e2cd",
    #     "#fdcdac",
    #     "#cbd5e8",
    #     "#f4cae4",
    #     "#e6f5c9",
    # ]

    # palette = [ # too bright
    #     "#A1C9F4",  # Pastel Blue
    #     "#FFB482",  # Pastel Orange
    #     "#8DE5A1",  # Pastel Green
    #     "#FF9F9B",  # Pastel Red
    #     "#D0BBFF",  # Pastel Purple
    #     "#FFFEA3",  # Pastel Yellow
    # ]

    # palette = [
    #     "#1F77B4",  # Cool Blue
    #     "#2CA02C",  # Green
    #     "#D62728",  # Warm Red
    #     "#9467BD",  # Purple
    #     "#FF7F0E",  # Orange
    #     "#17BECF",  # Teal
    # ]

    colors = {
        "S6": {"regex": ".*S6.*", "color": palette[0]},
        "p38": {"regex": ".*p38.*", "color": palette[1]},
        "T_STAT6": {"regex": ".*T.*STAT6.*", "color": palette[2]},
        "STAT6": {"regex": ".*STAT6.*", "color": palette[3]},
        "ERK": {"regex": ".*ERK.*", "color": palette[4]},
    }

    def nodes_node_color(g, n, d):
        for cname, cinfo in colors.items():
            if re.match(cinfo["regex"], feature_names[n]):
                return cinfo["color"]
        if n in rows and n in columns:
            return "black"
        else:
            return "grey"

    def edges_width(g, src, dst, d):
        abscor = abs(correlation_matrix[src, dst])
        if src == dst and not np.isclose(correlation_matrix[src,dst], 1):
            print(correlation_matrix[src,dst])
        t = 0.1
        if abscor < t:
            return 0
        else:
            return (abscor - t) / (1 - t) * 5

    def edges_edge_color(g, src, dst, d):
        cmap = sns.color_palette(palette="vlag", as_cmap=True)
        cor = correlation_matrix[src, dst]
        c = (cor + 1) / 2
        color = cmap(c)
        color = (*color[:-1], 0.7)
        return color
    
    ax.scatter(
        coo[:,0], coo[:,1], 
        color=[.5,.5,.5,.3], 
        linewidths=0, 
        edgecolors=None)
    
    nx_plot(
        # nodes=np.arange(coo.shape[0]),
        nodes_pos=nodes_pos,
        nodes_args=dict(
            node_size=nodes_node_size,
            node_color=nodes_node_color,
            edgecolors="black"
        ),
        edges=edges,
        edges_args=dict(
            width=edges_width,
            edge_color=edges_edge_color,
        ),
        # nodes_labels=np.arange(len(feature_names)),
        nodes_labels_args={
            "font_size": 6,
            "textcoords": 'offset points',
            "xytext": (5,0),
            "color": "black"
        },
        ax=ax)
    ax.set_title("A. Correlation network: " + timepoint_map_pp[cluster["reference_label"]], loc="left")
    ax.axis("off")
    
    ax = ax3
    ax.set_title("C. Disruption profile", loc='left')

    dis_values = disruption_matrices_dict[cluster["reference_label"]]
    dis_values = np.mean(dis_values[:,cluster["edges"].toarray() == 1], axis=(1))

    sns.boxplot(
        x=yy, 
        y=dis_values, 
        color="lightgrey",
        # # groups=entities,
        # groups_kwargs=dict(color="grey", linewidth=1, alpha=0.2),
        ax=ax)

    format_ax(ax)
    ax.set(xlabel="timepoint", ylabel="disruption", xticklabels=["healthy", "preeclamptic"])
    
#     ax = axes[row, 2]; ax.axis("off")
#     ax.set_title("B. Module features", loc='left')
#     offset = 0
#     for i, t in np.array(list(enumerate(feature_names)))[cluster["rows"]][idx_rows]:
#             ax.annotate(f"{int(i):3d}. {re.sub('^[0-9]* ', '', t.replace('_', ' '))}", (0, - offset), fontsize=5)
#             offset += 1
#     ax.set_ylim([-offset, 0])
    
    ax = ax2;
    ax.set_title("B. Correlation heatmap: " + timepoint_map_pp[cluster["reference_label"]], loc='left')
    rowcolors = [
        nodes_node_color(None, i, None) for i in cluster["rows"][idx_rows]]
    rowcolors = ["white" if r == "black" else r for r in rowcolors]
    sns.heatmap(
        correlation_matrices_dict[cluster["reference_label"]][cluster["rows"],:][:,cluster["columns"]][idx_rows,:][:,idx_cols],
        center=0, vmin=-1, vmax=1, cmap=cmap,
        xticklabels=np.array(feature_names)[cluster["rows"]][idx_rows],
        yticklabels=np.array([f"{i}_{f}   " for i,f in enumerate(feature_names)])[cluster["rows"]][idx_rows],
        cbar=False,
        ax=ax)
    # disable ticks
    ax.tick_params(left=False, bottom=False)
    
    for y, color in enumerate(rowcolors):
        ax.add_patch(plt.Rectangle(
            (-1, y),   # x, y in data coords
            -25,        # width (negative draws outside)
            1,           # height
            color=color,
            transform=ax.transData,
            clip_on=False
        ))
        ax.add_patch(plt.Rectangle(
            (y, len(rowcolors) + 3),   # x, y in data coords
            1,        # width (negative draws outside)
            -2,           # height
            color=color,
            transform=ax.transData,
            clip_on=False
        ))
    ax.annotate(
        "S6", 
        xy=(0,0), 
        xytext=(-28,6.5), 
        textcoords="data",  
        ha="right",
        # rotation=90,
        fontsize=14,
        bbox=dict(boxstyle="round,pad=0.2", fc=palette[0], ec="black", alpha=0.8))
    ax.annotate(
        "T-p38", 
        xy=(0,0), 
        xytext=(-28,22), 
        textcoords="data",  
        ha="right",
        # rotation=90,
        fontsize=14,
        bbox=dict(boxstyle="round,pad=0.2", fc=palette[1], ec="black", alpha=0.8))
    ax.annotate(
        "STAT6", 
        xy=(0,0), 
        xytext=(-28,31), 
        textcoords="data",
        ha="right",
        # rotation=90,
        fontsize=14,
        bbox=dict(boxstyle="round,pad=0.2", fc=palette[3], ec="black", alpha=0.8))
    ax.annotate(
        "T-ERK", 
        xy=(0,0), 
        xytext=(-28,39), 
        textcoords="data",
        ha="right",
        # rotation=90,
        fontsize=14,
        bbox=dict(boxstyle="round,pad=0.2", fc=palette[4], ec="black", alpha=0.8))
    ax.annotate(
        "T-STAT6", 
        xy=(0,0), 
        xytext=(-28,49.5), 
        textcoords="data",
        ha="right",
        # rotation=90,
        fontsize=14,
        bbox=dict(boxstyle="round,pad=0.2", fc=palette[2], ec="black", alpha=0.8))

    
    ticklabels = []
    for l in ax.get_yticklabels():
        arr = l.get_text().split("_")
        ticklabels.append(f"{' '.join(arr[1:]):>28s}")
        # ticklabels.append(f"{int(arr[0]):3d}. {' '.join(arr[1:]):>32s}")
    ax.set_yticklabels(ticklabels, fontsize = 5, family="monospace")
    ax.set_xticklabels(["" for _ in ax.get_xticklabels()], fontsize = 5)

    from matplotlib import cm
    ax = ax_r1_cb; ax.axis("off")
    fig.colorbar(cm.ScalarMappable(norm=Normalize(-1,1), cmap="vlag"), ax=ax, label="correlation", pad=0.3, location='left', fraction=1)
    ax = ax_r2_cb; ax.axis("off")
    fig.colorbar(cm.ScalarMappable(norm=norm, cmap="vlag"), ax=ax, label="disruption", pad=0.3, location='left', fraction=1)
        
    
    ax.annotate("D. Disruption networks", (0.1,0.50), xycoords="figure fraction", ha="left", fontsize=12)
    ax.annotate("E. Disruption heatmaps", (0.57,0.50), xycoords="figure fraction", ha="left", fontsize=12)
    
#     # sns.heatmap(
#     #     correlation_matrices_dict[cluster["reference_label"]][cluster["rows"],:][:,cluster["columns"]][idx_rows,:][:,idx_cols],
#     #     center=0, vmin=-1, vmax=1, cmap=cmap,
#     #     ax=ax)
    
    # for i, (key, _) in enumerate(coordinates_dict.items()):
    for i, key in enumerate([0, 1]):
        
        ax = ax_r2[i]
        ax.set_title(timepoint_map_pp[key])
        
        ax.axis("off")
        ax.scatter(coo[:,0], coo[:,1], color=[.5, .5, .5, .3])

        def edges_width(g, src, dst, d):
            dis = np.median(disruption_matrices[yy == key, src, dst])
            if True:\
                return np.abs((norm(dis) - 0.5)) * 2 * 10
            else:
                return .5

        def edges_edge_color(g, src, dst, d):
            dis = np.median(disruption_matrices[yy == key, src, dst])
            color = cmap(norm(dis))
            alpha = min(1, max(0, np.abs((norm(dis) - 0.5)) * 2))
            color = (*color[:-1], alpha)

            if True:
                return color
            else:
                return [0.7, 0.7, 0.7, 0.05]   

        nx_plot(
            # nodes=np.arange(coo.shape[0]),
            nodes_pos=nodes_pos,
            nodes_args=dict(
                node_size=nodes_node_size,
                node_color=nodes_node_color,
                edgecolors="black"
            ),
            edges=edges,
            edges_args=dict(
                width=edges_width,
                edge_color=edges_edge_color,
            ),
            # nodes_labels=labels,
            nodes_labels_args={
                "font_size": 2
            },
            ax=ax)
        
        
        # ax = axes[r, i]; r += 1
        # sns.heatmap(
        #     correlation_matrices_dict[key][cluster["rows"],:][:,cluster["columns"]][idx_rows,:][:,idx_cols],
        #     center=0, vmin=-1, vmax=1, cmap=cmap,
        #     xticklabels=cluster["rows"][idx_rows],
        #     yticklabels=cluster["rows"][idx_rows],
        #     ax=ax)
        
        ax = ax_r2[i + 2]
        ax.set_title(timepoint_map_pp[key])
        
        mean = np.mean(disruption_matrices[key == yy], axis=0)[cluster["rows"],:][:,cluster["columns"]]
        sns.heatmap(
            mean[idx_rows,:][:,idx_cols],
            center=0, vmin=-dis, vmax=dis, cmap=cmap,
            # xticklabels=cluster["rows"][idx_rows],
            # yticklabels=cluster["rows"][idx_rows],
            cbar=False,
            # cbar=(i == 3),
            ax=ax)
        
        # disable ticks
        ax.tick_params(left=False, bottom=False)
        # disable tick labels
        ax.set_xticklabels(["" for _ in ax.get_xticklabels()], fontsize = 5)
        ax.set_yticklabels(["" for _ in ax.get_yticklabels()], fontsize = 5)
        
        for y, color in enumerate(rowcolors):
            ax.add_patch(plt.Rectangle(
                (-1, y),   # x, y in data coords
                -2,        # width (negative draws outside)
                1,           # height
                color=color,
                transform=ax.transData,
                clip_on=False
            ))
            ax.add_patch(plt.Rectangle(
                (y, len(rowcolors) + 3),   # x, y in data coords
                1,        # width (negative draws outside)
                -2,           # height
                color=color,
                transform=ax.transData,
                clip_on=False
            ))
        
        # ax.set(xticklabels=np.arange(len(feature_names))[cluster["rows"]])
        ax.set_xticklabels(ax.get_xticklabels(), fontsize = 7)
        for tick in ax.xaxis.get_major_ticks()[1::2]:
            tick.set_pad(18)
            
        if i == 0:
            ax.set_yticklabels(ax.get_yticklabels(), fontsize = 7)
        else:
            ax.set_yticklabels(["" for _ in ax.get_yticklabels()], fontsize = 7)
        for tick in ax.yaxis.get_major_ticks()[1::2]:
            tick.set_pad(18)
        ax.set_xticklabels(["" for _ in ax.get_xticklabels()], fontsize = 5)
    
    fig.savefig(output_dir / f"cluster_classification_{i_cluster}_fancy.png", bbox_inches="tight")
    fig.savefig(output_dir / f"cluster_classification_{i_cluster}_fancy.pdf", bbox_inches="tight")

    if show_plots:
        plt.show()
        plt.tight_layout()
    else:
        plt.close()

plot_results(1, clusters[1], show_plots=True)

In [ ]:
import pandas as pd
records = []
for i, f in enumerate(feature_names):
    records.append((i, f.replace("_", " ")))
df = pd.DataFrame(records, columns=["id", "feature"])
df
print(df.to_latex(index=False, longtable=True))